# WM-Audit: 10-minute audit

Run the full battery on the built-in light env: train a tiny world model, compute pixel / control / uncertainty diagnostics, close-loop control, and correlate diagnostics with closed-loop return.

This tutorial run trains float baselines only (no compression lever), so the
preregistered H3 paired degradation test is honestly `not_assessable`; H1/H2
are still computed on the float rows.

In [ ]:
import sys, os
# Make the repo root importable whether the notebook is launched from
# notebooks/ (classic) or from the repo root (nbconvert / CI).
for _cand in (os.getcwd(), os.path.dirname(os.getcwd()), os.path.abspath('..')):
    if os.path.isfile(os.path.join(_cand, 'wm_audit', 'audit.py')):
        sys.path.insert(0, os.path.abspath(_cand))
        break
from wm_audit.audit import evaluate_config
from wm_audit.diagnostics import (
    audit_correlations,
    prereg_decision_h1, prereg_decision_h2, prereg_decision_h3,
    pixel_composite_scores, control_composite_scores,
)

In [ ]:
cfg = dict(buffer_capacity=2000, collect_steps=1500, data_seed=1,
           ensemble_size=2, train_steps=80, batch_size=32, lr=2e-3,
           calib_segments=12, calib_horizon=6, cf_pairs=20,
           cem_horizon=4, cem_candidates=24, cem_elites=6, cem_iters=2)
rows = []
for seed in [0, 1]:
    for hidden, embed in [(16, 16), (32, 16)]:
        # evaluate_config returns a LIST of rows (float baseline + any
        # compression variants); extend, do not append.
        rows.extend(evaluate_config('pixel_nav', hidden, embed, seed, cfg))
rows

In [ ]:
diag = ['pixel_mse','pixel_ssim','pixel_nll','return_ece',
        'counterfactual_tau','uncertainty_rho']
res = audit_correlations(rows, diag, return_key='R_real', n_boot=200)
for d, v in res.items():
    print(f"{d:20s} rho={v['rho']:+.3f} CI=[{v['ci_lo']:+.3f},{v['ci_hi']:+.3f}] p_adj={v['p_adj']:.3f}")

# Preregistered direction-unified composites on the FLOAT rows only (INT8
# rows are deployment perturbations and must never enter the H1/H2 pool),
# then the locked H1/H2/H3 verdicts (H3 needs compression rows; absent here
# -> not_assessable, never fabricated).
float_rows = [r for r in rows if not r.get('compression')]
for r, ps, cs in zip(float_rows, pixel_composite_scores(float_rows),
                     control_composite_scores(float_rows)):
    r['pixel_composite'] = float(ps)
    r['control_composite'] = float(cs)
cres = audit_correlations(float_rows, ['pixel_composite', 'control_composite'],
                          return_key='R_real', n_boot=200)
h1 = prereg_decision_h1(cres['pixel_composite']['rho'], threshold=0.4)
h2 = prereg_decision_h2(cres['control_composite']['rho'], threshold=0.6)
h3 = prereg_decision_h3(rows)
print('\nH1:', h1)
print('\nH2:', h2)
print('\nH3:', h3)